In [17]:
import openmc
import os

cross_sections_path = "/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml"

openmc.config["cross_sections"] = cross_sections_path

print("OpenMC version:", openmc.__version__)
print("Cross sections:", openmc.config["cross_sections"])
print("File exists:", os.path.exists(cross_sections_path))

OpenMC version: 0.16.0
Cross sections: /home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml
File exists: True


In [37]:
# Cell 2: materials
# graphite = ...
# salt = ...
# inconel = ...
# inor = ...
# bush = ...

inconel.name = "inconel-600"
inor.name = "inor-8"
bush.name = "Gd2O3-Al2O3"

materials = openmc.Materials([
    helium, salt, graphite, inconel, inor, bush
])

materials.export_to_xml()

In [35]:
# ============================================================
# MSRE CONTROL ROD SETUP
# Run this after helium/salt/graphite/inconel/inor are defined
# ============================================================

import openmc
import os


# ------------------------------------------------------------
# 1. Cross section library
# ------------------------------------------------------------

openmc.config["cross_sections"] = (
    "/home/linz/ondemand/openmc/nuclear_data/"
    "endfb-viii.0-hdf5/cross_sections.xml"
)


# ------------------------------------------------------------
# 2. Clean old XML files
# ------------------------------------------------------------

os.system("rm -f materials.xml geometry.xml tallies.xml")


# ------------------------------------------------------------
# 3. DAGMC geometry
# ------------------------------------------------------------

h5m_filepath = "../h5m/msre_control_rod.h5m"

dag_univ = openmc.DAGMCUniverse(h5m_filepath)

# OpenMC 0.16 needs an outer boundary condition
bounded_univ = dag_univ.bounded_universe()

geom = openmc.Geometry(root=bounded_univ)
geom.export_to_xml()


# ------------------------------------------------------------
# 4. Control rod absorber material: Gd2O3-Al2O3
# ------------------------------------------------------------

# Weight fractions
b_w_conc = {
    "Al2O3": 0.3,
    "Gd2O3": 0.7
}

# Atomic weights [g/mol]
A_w = {
    "Al": 26.9815385,
    "Gd": 157.25,
    "O": 15.99
}

# Densities [g/cc]
rho = {
    "Al2O3": 3.987,
    "Gd2O3": 7.07
}

# Molecular weights
b_mol_w = {
    "Al2O3": 2 * A_w["Al"] + 3 * A_w["O"],
    "Gd2O3": 2 * A_w["Gd"] + 3 * A_w["O"]
}

# Convert weight composition to molecular composition
denominator = (
    b_w_conc["Al2O3"] / b_mol_w["Al2O3"]
    + b_w_conc["Gd2O3"] / b_mol_w["Gd2O3"]
)

bush_mol_comp = {
    "Al2O3":
        (b_w_conc["Al2O3"] / b_mol_w["Al2O3"]) / denominator,

    "Gd2O3":
        (b_w_conc["Gd2O3"] / b_mol_w["Gd2O3"]) / denominator
}

# IMPORTANT:
# The material name must exactly match the DAGMC tag
bush = openmc.Material(name="Gd2O3-Al2O3")

bush.add_element(
    "Al",
    2 / 5 * bush_mol_comp["Al2O3"],
    "ao"
)

bush.add_element(
    "Gd",
    2 / 5 * bush_mol_comp["Gd2O3"],
    "ao"
)

bush.add_element(
    "O",
    (
        3 / 5 * bush_mol_comp["Al2O3"]
        + 3 / 5 * bush_mol_comp["Gd2O3"]
    ),
    "ao"
)

bush.set_density(
    "g/cc",
    (
        b_w_conc["Al2O3"] * rho["Al2O3"]
        + b_w_conc["Gd2O3"] * rho["Gd2O3"]
    )
)


# ------------------------------------------------------------
# 5. Make existing material names match DAGMC tags
# ------------------------------------------------------------

inconel.name = "inconel-600"
inor.name = "inor-8"


# ------------------------------------------------------------
# 6. Export materials
# ------------------------------------------------------------

materials = openmc.Materials([
    helium,
    salt,
    graphite,
    inconel,
    inor,
    bush
])

materials.export_to_xml()


# ------------------------------------------------------------
# 7. Sanity check
# ------------------------------------------------------------

print("DAGMC requires:")
print(dag_univ.material_names)

print("\nOpenMC materials:")
for mat in materials:
    print(mat.id, mat.name)

print("\nGeometry file:")
print(h5m_filepath)

print("\nCross sections:")
print(openmc.config["cross_sections"])

DAGMC requires:
['Gd2O3-Al2O3', 'inconel-600', 'inor-8']

OpenMC materials:
10 helium
8 salt
6 graphite
9 inconel-600
7 inor-8
11 Gd2O3-Al2O3

Geometry file:
../h5m/msre_control_rod.h5m

Cross sections:
/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml


/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10000.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10001.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10002.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10003.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10004.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Surface instance already exists with id=10005.
  warn(msg, IDWarning)
/opt/python/lib64/python3.12/site-packages/openmc/mixin.py:72: IDWarning: Another Cell i

In [36]:
# ============================================================
# MESH SETUP
# ============================================================

mesh1 = openmc.RegularMesh()
mesh1.dimension = (400, 400, 1)
mesh1.lower_left = (-175, -175, 100)
mesh1.upper_right = (175, 175, 101)

mesh1_filter = openmc.MeshFilter(mesh1)


mesh2 = openmc.RegularMesh()
mesh2.dimension = (400, 1, 400)
mesh2.lower_left = (-175, 0, -50)
mesh2.upper_right = (175, 1, 350)

mesh2_filter = openmc.MeshFilter(mesh2)

In [38]:
# ============================================================
# TALLIES
# ============================================================

t1 = openmc.Tally(name="flux1")
t1.filters = [mesh1_filter]
t1.scores = [
    "flux",
    "fission",
    "absorption"
]

t2 = openmc.Tally(name="flux2")
t2.filters = [mesh2_filter]
t2.scores = [
    "flux",
    "fission",
    "absorption"
]

tallies = openmc.Tallies([t1, t2])
tallies.export_to_xml()

In [39]:
# ============================================================
# SOURCE + SETTINGS
# ============================================================

source_volume = openmc.stats.Box(
    [-125, -125, 0],
    [125, 125, 500],
    only_fissionable=True
)

source = openmc.Source(space=source_volume)
source.angle = openmc.stats.Isotropic()

settings = openmc.Settings()

settings.source = source
settings.batches = 20
settings.inactive = 5
settings.particles = 20000

# Adjust only if needed / approved
settings.source_rejection_fraction = 0.01

settings.export_to_xml()

/opt/python/lib64/python3.12/site-packages/openmc/stats/multivariate.py:1115: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "
/opt/python/lib64/python3.12/site-packages/openmc/source.py:668: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn(


In [34]:
# Cell 3: geometry
h5m_filepath = "/home/linz/ondemand/openmc/msre/msre_docker/msre_control_in.h5m"

dag_univ = openmc.DAGMCUniverse(h5m_filepath)

bounded_univ = dag_univ.bounded_universe()

geom = openmc.Geometry(root=bounded_univ)
geom.export_to_xml()

In [18]:
import os

workdir = "/home/linz/ondemand/openmc/msre/emily_tutorial"
os.makedirs(workdir, exist_ok=True)
os.chdir(workdir)

print("Working directory:", os.getcwd())

Working directory: /home/linz/ondemand/openmc/msre/emily_tutorial


In [19]:
print("Cross sections:", openmc.config['cross_sections'])

Cross sections: /home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml


In [20]:
"/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml"

'/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'

In [21]:
import openmc

openmc.config['cross_sections'] = '/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'

print("Cross sections:", openmc.config['cross_sections'])

Cross sections: /home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml


In [22]:
import os

print("Home:", os.path.expanduser("~"))
print("Current directory:", os.getcwd())
print("OpenMC folder exists:", os.path.exists("/home/linz/ondemand/openmc"))
print("Cross sections exists:", os.path.exists("/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml"))

Home: /home/linz
Current directory: /home/linz/ondemand/openmc/msre/emily_tutorial
OpenMC folder exists: True
Cross sections exists: True


In [23]:
import os

for path in ["/apps", "/hpc-common", "/projects", "/scratch", "/ibmstor", "/home/linz"]:
    print(path, "->", os.path.exists(path))

/apps -> True
/hpc-common -> True
/projects -> True
/scratch -> True
/ibmstor -> True
/home/linz -> True


In [24]:
!find /home/linz/ondemand/openmc -name "cross_sections.xml" 2>/dev/null

/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml


In [25]:
openmc.config['cross_sections'] = '/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'

print(openmc.config['cross_sections'])

/home/linz/ondemand/openmc/nuclear_data/endfb-viii.0-hdf5/cross_sections.xml


In [26]:
import os
print(os.path.exists(openmc.config['cross_sections']))

True


In [27]:
"/home/linz/ondemand/openmc/msre/msre_docker/MSRE.ipynb"

'/home/linz/ondemand/openmc/msre/msre_docker/MSRE.ipynb'

In [28]:
import openmc

In [29]:
graphite=openmc.Material(name='graphite')
graphite.add_element('C',1.0,'ao')
graphite.set_density('g/cc',2.26)

#Hastelloy N / INOR-8 nominal material composition from
#ORNL-TM-4189
inor=openmc.Material(name='inor')
inor.add_element('Ni',0.72)
inor.add_element('Mo',0.16)
inor.add_element('Cr',0.07)
inor.add_element('Fe',0.05)
inor.set_density('g/cc',9)

# LiF,BeF2,UF4,ZrF4 [0.67,0.23,0.05,0.0079] mol % @ 33% enrichment 
molar_comp={'LiF':0.67,'BeF2':0.23, 'ZrF4':0.05, 'UF4':0.0079}
enrichment=0.3333
# MYYYYYYYYNOTE:
# Verify the original MSRE fuel-salt composition.
# The listed fractions sum to 0.9579 rather than 1.0,
# and the UF4/ZrF4 ordering in the comment differs from the code.
salt=openmc.Material(name='salt')
salt.add_element('F',molar_comp['LiF']*1/2+molar_comp['BeF2']*2/3+molar_comp['ZrF4']*4/5+molar_comp['UF4']*4/5,'ao')
salt.add_nuclide('Li7',molar_comp['LiF']*1/2,'ao')
salt.add_element('Be',molar_comp['BeF2']*1/3,'ao')
salt.add_element('Zr',molar_comp['ZrF4']*1/5,'ao')
salt.add_nuclide('U235',enrichment*molar_comp['UF4']*1/5,'ao')
salt.add_nuclide('U238',(1-enrichment)*molar_comp['UF4']*1/5,'ao')
salt.set_density('g/cc',2.2)

# The natural isotopes have been used for this alloy
# The density is set to that of Ni.
inconel=openmc.Material(name='inconel')
inconel.add_element('Ni',0.72,'ao')
inconel.add_element('Cr',0.20,'ao')
inconel.add_element('Fe',0.08,'ao')
inconel.set_density('g/cc',8.9)

helium=openmc.Material(name='helium')
helium.add_nuclide('He4',1.0,'ao')
helium.set_density('g/cc',1.0e-4)

materials=openmc.Materials([helium,salt,graphite,inconel,inor])
materials.export_to_xml()

In [30]:
materials

[Material
 	ID             =	10
 	Name           =	helium
 	Temperature    =	None
 	Density        =	0.0001 [g/cc]
 	Volume         =	None [cm^3]
 	Depletable     =	False
 	S(a,b) Tables  
 	Nuclides       
 	He4            =	1.0          [ao],
 Material
 	ID             =	8
 	Name           =	salt
 	Temperature    =	None
 	Density        =	2.2 [g/cc]
 	Volume         =	None [cm^3]
 	Depletable     =	True
 	S(a,b) Tables  
 	Nuclides       
 	F19            =	0.5346533333333334 [ao]
 	Li7            =	0.335        [ao]
 	Be9            =	0.07666666666666667 [ao]
 	Zr90           =	0.005144999999999999 [ao]
 	Zr91           =	0.001122     [ao]
 	Zr92           =	0.0017150000000000002 [ao]
 	Zr94           =	0.0017380000000000002 [ao]
 	Zr96           =	0.00028000000000000003 [ao]
 	U235           =	0.000526614  [ao]
 	U238           =	0.0010533860000000003 [ao],
 Material
 	ID             =	6
 	Name           =	graphite
 	Temperature    =	None
 	Density        =	2.26 [g/cc]
 	Volume    

In [31]:
h5m_filepath = "/home/linz/ondemand/openmc/msre/msre_docker/msre_simple.h5m"

dag_univ = openmc.DAGMCUniverse(h5m_filepath)
geom = openmc.Geometry(root=dag_univ)
geom.export_to_xml()

In [32]:
xwidth = 350
yheight = 350
material_colors={salt:'red', inor:'lightblue', inconel:'blue',helium:'white',graphite:'gray'}
#xz plot
p1 = openmc.Plot()
p1.background='white'
p1.basis = 'xz'
p1.width = (xwidth,yheight)
p1.origin=(0,0,125)
p1.pixels = (4000, 4000)
p1.color_by = 'material'
p1.colors=material_colors
#xy plot
p2 = openmc.Plot()
p2.background='white'
p2.basis='xy'
p2.width=(xwidth, yheight)
p2.pixels = (4000,4000)
p2.origin=(0,0,100)
p2.color_by='material'
p2.colors=material_colors

plots=openmc.Plots([p1,p2])
openmc.plot_inline(plots)

/opt/python/lib64/python3.12/site-packages/openmc/plots.py:1389: FutureWarning: The Plot class is deprecated. Use SlicePlot for 2D slice plots or VoxelPlot for 3D voxel plots.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# High-resolution XY plot of the MSRE central control-rod region

p_control = openmc.Plot()

p_control.background = 'white'
p_control.basis = 'xy'

# Match the reference cross section
p_control.origin = (0, 0, 163)

# Show only the central region containing the 3 control rods
# and the sample assembly
p_control.width = (25, 25)

# High resolution
p_control.pixels = (2500, 2500)

p_control.color_by = 'material'
p_control.colors = material_colors

control_plots = openmc.Plots([p_control])

openmc.plot_inline(control_plots)｜

In [44]:
# Create a neutron source for kick-starting
source_volume=openmc.stats.Box([-125,-125,0],[125,125,500], only_fissionable=True)
source = openmc.Source(space=source_volume)
source.angle=openmc.stats.Isotropic()

/opt/python/lib64/python3.12/site-packages/openmc/stats/multivariate.py:1115: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "
/opt/python/lib64/python3.12/site-packages/openmc/source.py:668: FutureWarning: This class is deprecated in favor of 'IndependentSource'
  warnings.warn(


In [43]:
settings = openmc.Settings()
settings.source = source

settings.batches = 20
settings.inactive = 5
settings.particles = 20000

# Allow a lower source sampling acceptance fraction
settings.source_rejection_fraction = 0.01

settings.export_to_xml()

openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

KeyboardInterrupt: 

In [ ]:
mesh1=openmc.RegularMesh()
mesh1.dimension =  [400,400,1]
mesh1.lower_left = [-125, -125, 90]
mesh1.upper_right = [125, 125, 110]
mesh1_filter = openmc.MeshFilter(mesh1)

mesh2=openmc.RegularMesh()
mesh2.dimension =  [400,1,400]
mesh2.lower_left = [-175, -10, -50]
mesh2.upper_right = [175,  10, 400]
mesh2_filter = openmc.MeshFilter(mesh2)

t1 = openmc.Tally(name='flux1')
t1.filters =[mesh1_filter]
t1.scores = ['flux','fission']

t2 = openmc.Tally(name='flux2')
t2.filters =[mesh2_filter]
t2.scores = ['flux','fission']

tallies=openmc.Tallies([t1,t2])
tallies.export_to_xml()

In [ ]:
import os
os.system("rm -f summary.h5 statepoint.20.h5")
openmc.run()

In [ ]:
sp=openmc.StatePoint('statepoint.20.h5')

tl1=sp.get_tally(name='flux1')
tl2=sp.get_tally(name='flux2')

flux1=tl1.get_slice(scores=['flux'])
flux2=tl2.get_slice(scores=['flux'])

flux1.mean.shape=(400,400)
flux2.mean.shape=(400,400)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig,(ax1,ax2)=plt.subplots(ncols=2,figsize=(20,16), constrained_layout=True)

ax1.set_xticks(np.arange(0,401,399/4))
ax1.set_xticklabels(np.arange(-175,176,350./4))
ax2.set_xticks(np.arange(0,400,399/4))
ax2.set_xticklabels(np.arange(-175,176,350./4))
ax1.set_yticks(np.arange(0,400,399/4))
ax1.set_yticklabels(np.arange(-175,176,350./4))
ax2.set_yticks(np.arange(0,400,399/4))
ax2.set_yticklabels(np.arange(-175,176,350./4))
ax1.set_xlabel('X / cm')
ax1.set_ylabel('Y / cm')

ax2.set_xlabel('X / cm')
ax2.set_ylabel('Z / cm')
im1=ax1.imshow(flux1.mean)
fig.colorbar(im1,ax=ax1,shrink=0.4)
im2=ax2.imshow(flux2.mean)
fig.colorbar(im2,ax=ax2,shrink=0.4)

In [ ]:
#Similarly we can plot fission reactions:

fission1=tl1.get_slice(scores=['fission'])
fission2=tl2.get_slice(scores=['fission'])

fission1.mean.shape=(400,400)
fission2.mean.shape=(400,400)

fig,(ax1,ax2)=plt.subplots(ncols=2,figsize=(20,16), constrained_layout=True)

ax1.set_xticks(np.arange(0,401,399/4))
ax1.set_xticklabels(np.arange(-175,176,350./4))
ax2.set_xticks(np.arange(0,400,399/4))
ax2.set_xticklabels(np.arange(-175,176,350./4))
ax1.set_yticks(np.arange(0,400,399/4))
ax1.set_yticklabels(np.arange(-175,176,350./4))
ax2.set_yticks(np.arange(0,400,399/4))
ax2.set_yticklabels(np.arange(-175,176,350./4))
ax1.set_xlabel('X / cm')
ax1.set_ylabel('Y / cm')

ax2.set_xlabel('X / cm')
ax2.set_ylabel('Z / cm')
im1=ax1.imshow(fission1.mean)
fig.colorbar(im1,ax=ax1,shrink=0.4)
im2=ax2.imshow(fission2.mean)
fig.colorbar(im2,ax=ax2,shrink=0.4)

In [ ]:
#define a lograrithmic binning from 1keV to 10 MeV
ef=energyrange=openmc.EnergyFilter(np.logspace(3,7,200))
te = openmc.Tally(name='energy')

sf = openmc.MaterialFilter(salt)
te.filters =[ef,sf]
te.scores = ['flux','fission']

tallies.append(te)
tallies.export_to_xml()

In [ ]:
os.system("rm -f summary.h5 statepoint.20.h5")
openmc.run()

In [ ]:
sp=openmc.StatePoint('statepoint.20.h5')

In [ ]:
tl=sp.get_tally(name='energy')

e1=tl.get_slice(scores=['flux'])
e2=tl.get_slice(scores=['fission'])

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.5),squeeze=True)
ax.plot(ef.values[:199],e1.mean[:,0,0])
ax.plot(ef.values[:199],e2.mean[:,0,0])
ax.set_xscale('log')
ax.set_yscale('log')

In [ ]:
#geometry
os.system('rm -f materials.xml geometry.xml tallies.xml')
h5m_filepath="../h5m/msre_control_rod.h5m"
dag_univ = openmc.DAGMCUniverse(h5m_filepath)
geom = openmc.Geometry(root=dag_univ)
geom.export_to_xml()

b_w_conc={'Al2O3':0.3,'Gd2O3':0.7} # Wt. (D. Shen et.al. Nucl. Sc. & Eng., v. 195, pp. 825, 2021)
A_w={'Al':26.9815385,'Gd':157.25, 'O':15.99} #g/mol, (Webelements.com)
rho={'Al2O3':3.987,'Gd2O3':7.07} # g /cc, (Wikipedia: Aluminium_oxide & Gadolinium(III)_oxide)
b_mol_w={'Al2O3':2*A_w['Al']+ 3*A_w['O'],'Gd2O3':2*A_w['Gd']+3*A_w['O']}
bush_mol_comp={'Al2O3':(b_w_conc['Al2O3']/b_mol_w['Al2O3'])/( b_w_conc['Al2O3']/b_mol_w['Al2O3'] + b_w_conc['Gd2O3']/b_mol_w['Gd2O3'] ),
                'Gd2O3':(b_w_conc['Gd2O3']/b_mol_w['Gd2O3'])/( b_w_conc['Al2O3']/b_mol_w['Al2O3'] + b_w_conc['Gd2O3']/b_mol_w['Gd2O3'] )}

bush=openmc.Material(name='bush')
bush.add_element('Al',2/5*bush_mol_comp['Al2O3'],'ao')
bush.add_element('Gd',2/5*bush_mol_comp['Gd2O3'],'ao')
bush.add_element('O',3/5*bush_mol_comp['Al2O3']+3/5*bush_mol_comp['Gd2O3'],'ao')
bush.set_density('g/cc',b_w_conc['Al2O3']*rho['Al2O3'] +b_w_conc['Gd2O3']*rho['Gd2O3'] )

t1 = openmc.Tally(name='flux1')
t1.filters =[mesh1_filter]
t1.scores = ['flux','fission','absorption']

t2 = openmc.Tally(name='flux2')
t2.filters =[mesh2_filter]
t2.scores = ['flux','fission','absorption']

tallies=openmc.Tallies([t1,t2])
tallies.export_to_xml()


materials=openmc.Materials([helium,salt,graphite,inconel,inor,bush])
materials.export_to_xml()

In [ ]:
xwidth = 350
yheight = 350
material_colors={salt:'red', inor:'lightblue', inconel:'blue',helium:'white',graphite:'gray',bush:'purple'}
#xz plot
p1 = openmc.Plot()
p1.background='white'
p1.basis = 'xz'
p1.width = (xwidth,yheight)
p1.origin=(0,0,125)
p1.pixels = (800, 800)
p1.color_by = 'material'
p1.colors=material_colors
#xy plot
p2 = openmc.Plot()
p2.background='white'
p2.basis='xy'
p2.width=(xwidth, yheight)
p2.pixels = (800,800)
p2.origin=(0,0,100)
p2.color_by='material'
p2.colors=material_colors

p3 = openmc.Plot()
p3.background='white'
p3.basis='xy'
p3.width=(xwidth/10, yheight/10)
p3.pixels = (800,800)
p3.origin=(0,0,100)
p3.color_by='material'
p3.colors=material_colors

plots=openmc.Plots([p1,p2,p3])
openmc.plot_inline(plots)

In [ ]:
print(dag_univ.material_names)

In [ ]:
for mat in materials:
    print(mat.id, mat.name)

In [ ]:
inconel.name = 'inconel-600'
inor.name = 'inor-8'

In [ ]:
bush.name = 'Gd2O3-Al2O3'

In [ ]:
openmc.plot_inline(plots)

In [ ]:
print(os.path.exists("geometry.xml"))
print(os.path.exists("materials.xml"))
print(os.path.exists("plots.xml"))

In [ ]:
materials.export_to_xml()

In [ ]:
# Make sure the CURRENT material objects have the exact DAGMC names
inconel.name = 'inconel-600'
inor.name = 'inor-8'
bush.name = 'Gd2O3-Al2O3'

# Rebuild the Materials collection using THESE exact objects
materials = openmc.Materials([
    helium,
    salt,
    graphite,
    inconel,
    inor,
    bush
])

print("Materials being exported:")
for mat in materials:
    print(mat.id, mat.name)

# Export
materials.export_to_xml()

In [ ]:
with open("materials.xml", "r") as f:
    text = f.read()

print(text)

In [ ]:
openmc.plot_inline(plots)

In [ ]:
bounded_univ = dag_univ.bounded_universe()
geom = openmc.Geometry(root=bounded_univ)
geom.export_to_xml()

In [45]:
os.system("rm -f summary.h5 statepoint.20.h5")
settings.source_rejection_fraction = 0.01

settings.export_to_xml()
openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

RuntimeError: Exceeded maximum number of source rejections per sample. Please check your source definition. -------------------------------------------------------------------------- MPI_ABORT was invoked on rank 0 in communicator MPI_COMM_WORLD Proc: [[43808,0],0] Errorcode: -1 NOTE: invoking MPI_ABORT causes Open MPI to kill all MPI processes. You may or may not see output from other processes, depending on exactly when Open MPI kills them. --------------------------------------------------------------------------